## 欢迎来到第四周第四天

这是一个非常棒的项目！简单但非常有效。

In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.prebuilt import ToolNode, tools_condition
import requests
import os
from langchain.agents import Tool

from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
load_dotenv(override=True)

### 异步 LangGraph

运行一个工具：  
同步: `tool.run(inputs)`  
异步: `await tool.arun(inputs)`

调用 graph：  
同步: `graph.invoke(state)`  
异步: `await graph.ainvoke(state)`

In [3]:
class State(TypedDict):
    
    messages: Annotated[list, add_messages]


graph_builder = StateGraph(State)

In [4]:
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(text: str):
    """Send a push notification to the user"""
    requests.post(pushover_url, data = {"token": pushover_token, "user": pushover_user, "message": text})

tool_push = Tool(
        name="send_push_notification",
        func=push,
        description="useful for when you want to send a push notification"
    )

## 额外的安装步骤——如果你的电脑上没有 Node 和 Playwright

接下来，如果你的电脑上还没有安装 NodeJS 和 Playwright，需要先安装它们。请参考以下说明：

[Node 和 Playwright 安装指南](../setup/SETUP-node.md)

## 注意——安装 Playwright 后，Windows PC 用户请看这里：

在执行下面几个 cell 时，你可能会遇到 Playwright 浏览器抛出 NotImplementedError 的问题。

当我们迁移到 Python 模块时应该能正常工作，但在 Windows 的 notebook 中可能会出问题。

如果你遇到了这个错误，但仍想在 notebook 中运行，你需要做一个小小的修改（看起来有些 hacky！）。你需要在安装 Playwright（前面的 cell）之后做以下操作：

1. 右键点击左侧文件浏览器中的 `.venv` 文件夹，选择"在文件夹中查找"
2. 搜索 `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())`  
3. 这段代码应该出现在一个叫 `kernelapp.py` 的文件中
4. 将这段代码所在的整个 else 子句注释掉——参考下面的代码片段。确保 ImportError 行之后有 "pass" 语句。
5. 点击上方的 "Restart" 按钮重启内核

```python
        if sys.platform.startswith("win") and sys.version_info >= (3, 8):
            import asyncio
 
            try:
                from asyncio import WindowsProactorEventLoopPolicy, WindowsSelectorEventLoopPolicy
            except ImportError:
                pass
                # 不受影响
           # else:
            #    if type(asyncio.get_event_loop_policy()) is WindowsProactorEventLoopPolicy:
                    # WindowsProactorEventLoopPolicy 不兼容 tornado 6
                    # 回退到 Python 3.8 之前的默认 Selector
                    # asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())
```

感谢学员 Nicolas 发现此问题，以及 Kalyan、Yaki、Zibin 和 Bhaskar 确认这个方法有效！也感谢 Vladislav 提供的额外提示。

作为替代方案，你可以直接迁移到 Python 模块（我们第 5 天本来就会这样做）。

In [ ]:
# 引入 nest_asyncio
# Python 的异步代码只允许一个"事件循环"来处理异步事件。
# `nest_asyncio` 库可以修补这个限制，适用于需要运行嵌套事件循环的特殊场景。

import nest_asyncio
nest_asyncio.apply()

### LangChain 社区

LangChain 最引人注目的特性之一就是它丰富的社区生态。

看看这个：

In [ ]:
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser

# 如果在这里或后续步骤中遇到 NotImplementedError，请参考本 notebook 顶部的"注意"说明
# 在本机后台启动一个看不见的游览器
async_browser =  create_async_playwright_browser(headless=False)  # headful 模式（可见浏览器窗口）
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
tools = toolkit.get_tools() #返回的是 一组能操控真实浏览器的 LangChain Tool 对象

In [ ]:
for tool in tools:
    print(f"{tool.name}={tool}")

In [ ]:
tool_dict = {tool.name:tool for tool in tools}
# 这里通过工具名称获取到对应的工具对象，方便后续调用
navigate_tool = tool_dict.get("navigate_browser")
extract_text_tool = tool_dict.get("extract_text")

# 游览器的操作需要等待天生就是异步的
await navigate_tool.arun({"url": "https://www.cnn.com"})
text = await extract_text_tool.arun({})

In [ ]:
import textwrap
print(textwrap.fill(text))

In [17]:
all_tools = tools + [tool_push]

In [18]:

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(all_tools)


def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


In [ ]:

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=all_tools))
graph_builder.add_conditional_edges( "chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "10"}}

async def chat(user_input: str, history):
    result = await graph.ainvoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    return result["messages"][-1].content


gr.ChatInterface(chat, type="messages").launch()